# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates step-by-step how to load, inspect, and process the FAIR² dataset using the `mlcroissant` library. All entities (record sets, fields, columns) are referenced by their `@id` values. 

### Dataset Source
The dataset source is provided via a Croissant schema URL:
<https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json>


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(url)

# Print core metadata
print("Dataset loaded.")
print(f"Name: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}")
print(f"Version: {dataset.metadata.version}")
print(f"License: {dataset.metadata.license}")

## 2. Data Overview
Review available record sets and fields, referencing their `@id`s as defined by the Croissant schema.

We'll use the Croissant schema structure to inspect top-level record sets, their fields, and columns. All references use their exact `@id`.

In [ ]:
# Helper: print available record sets and their fields/columns by @id
record_sets = list(dataset.record_sets)
print(f"Found {len(record_sets)} record set(s):\n")
for rs in record_sets:
    print(f"RecordSet @id: {rs.id}")
    print(f"  Name: {rs.name}")
    print(f"  Description: {rs.description}")
    print("  Fields and columns:")
    # Print field @ids
    for field in rs.fields:
        print(f"    Field @id: {field.id} Name: {field.name} (type: {field.data_type})")
        for column in getattr(field, 'columns', []):
            print(f"      Column @id: {column.id} Name: {column.name}")
    print()
# For each record set, print a sample record
for rs in record_sets:
    print(f"Sample record from RecordSet @id: {rs.id}")
    records = list(dataset.records(record_set=rs.id))
    if records:
        print(records[0])
    else:
        print("  No records available.")


## 3. Data Extraction
Load data from each record set into a pandas DataFrame for analysis. Use record set and field `@id`s as listed above.


In [ ]:
# Create DataFrames for each RecordSet, referenced by their @id
dataframes = {}
for rs in record_sets:
    records = list(dataset.records(record_set=rs.id))
    dataframes[rs.id] = pd.DataFrame(records)
    print(f"Loaded {len(records)} records for RecordSet @id: {rs.id}")

# Show columns for each DataFrame
for rs in record_sets:
    print(f"\nColumns for RecordSet @id: {rs.id}")
    print(dataframes[rs.id].columns.tolist())
    display(dataframes[rs.id].head())

## 4. Exploratory Data Analysis (EDA)
Apply exploratory and processing steps for the selected record set – filtering, normalizing, grouping. Reference each field using their `@id` and use variables for repeat operations.


In [ ]:
# Example: Select a numeric field by @id from the first non-empty record set for demonstrations
from numpy import number

# Helper to find a numeric field
target_record_set_id = None
numeric_field_id = None

# Find a record set with numeric data
for rs in record_sets:
    df = dataframes[rs.id]
    if len(df) == 0:
        continue
    for col in df.columns:
        # Try to interpret as numeric, allow errors
        sample = pd.to_numeric(df[col], errors='coerce')
        if sample.notnull().sum() > 0:
            numeric_field_id = col
            target_record_set_id = rs.id
            break
    if numeric_field_id:
        break
if target_record_set_id is None:
    print("No numeric field found in any recordset.")
else:
    print(f"Using RecordSet @id: {target_record_set_id}")
    print(f"Using numeric field: {numeric_field_id}")

    df = dataframes[target_record_set_id].copy()
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')

    # Filter: only keep records where the numeric value > threshold
    threshold = df[numeric_field_id].mean() if df[numeric_field_id].notnull().sum() else 10
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}")
    display(filtered_df[[numeric_field_id]].head())

    # Normalize field
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id}:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by a candidate categorical field if it exists
    group_field = None
    for col in filtered_df.columns:
        if col == numeric_field_id:
            continue
        # Check if dtype is not numeric
        if filtered_df[col].dtype == 'object':
            group_field = col
            break
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
        print(f"Grouped data by {group_field} (showing group mean of {numeric_field_id}):")
        display(grouped_df.head())
    else:
        print("No categorical field found for grouping.")

## 5. Visualization
Visualize the data distributions or relationships between numeric and (if present) group/categorical fields. This section requires `matplotlib` and/or `seaborn`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if target_record_set_id and numeric_field_id:
    plt.figure(figsize=(8, 5))
    sns.histplot(data=filtered_df, x=numeric_field_id, kde=True)
    plt.title(f"Distribution of {numeric_field_id} in RecordSet {target_record_set_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    if 'group_field' in locals() and group_field:
        plt.figure(figsize=(10, 5))
        sns.boxplot(x=group_field, y=numeric_field_id, data=filtered_df)
        plt.title(f"{numeric_field_id} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
This exercise demonstrated loading and exploring the FAIR² record-oriented mass spectrometry dataset using the Croissant schema and `mlcroissant` library. We referenced every entity by its `@id`, inspected available record sets and fields, extracted records into pandas DataFrames, and performed basic exploratory data analysis and visualization. 

Using the Croissant approach ensures field and file identity is unambiguous, making robust data pipelines and FAIR sharing easy to maintain for downstream analysis or publication.